# 05 · Simulation outputs: from TNG snapshots to gas fields

`skewerjax.simulations` turns the outputs of a hydrodynamical simulation (here a TNG snapshot) into the gas along the
skewers that the optical depth needs. It is organised in layers:

| layer | module | does | arrays |
|---|---|---|---|
| 1 | `readers` | HDF5 I/O and units (`ArepoHDF5Reader`) | NumPy |
| 2 | `gas_cells` | `GasCells`: positions, volumes, physical fields | pytree |
| 3 | `gas_physics` | $n_{\rm H}$, $T$, $x_{\rm HI}$, metal ion fractions (CLOUDY) per cell | JAX |
| 4 | `snapshot_cells`, `selection` | the cells of a region of the box, or those that can reach a set of skewers (periodic k-d tree, streamed) | NumPy (CPU); optional GPU back end |
| 5 | `snapshot_cells`, `SimulationSkewers.to_cells` | the cells as `core.Cells`: per ion, comoving density, temperature, peculiar velocity | NumPy |

The last layer produces the same container as the toy models of notebook 04. From there the route to $\tau$ belongs to
the core and is the same for every source: deposition of the cells into the pieces of the skewers, and the sum over
the pieces (notebook 07).

Why the remaining NumPy steps are NumPy:

- **Reader.** The snapshot is read from HDF5 in streamed blocks (h5py); that is host I/O.
- **Selection.** It is a periodic k-d tree query (`scipy.spatial.cKDTree(boxsize=...)`) of every streamed block of cells
  against the path points.
  - Its output size, the number of selected cells, depends on the data, which static JAX shapes cannot express. It is
    pure geometry and is never differentiated.
  - `select_cells_near_paths(..., backend=...)` and `SimulationSkewers(..., selection_backend=...)` take:
    - `"cpu"` (the default);
    - `"jztree"`, a CUDA-only nearest-neighbour search with the optional package JZ-TREE, with borderline cells
      re-checked in float64 so the selected set is the same;
    - `"auto"`, which uses the GPU when that package and a CUDA device are present.
`SimulationSkewers` runs these layers, and the core's route after them, for any `ExtractionOptions` and
`IntegrationOptions`.

The data are TNG50-4 snapshot 25 ($z = 3.0$, 35 cMpc$/h$, 19.3 M cells), read-only. Runtime: ~1 min. Notebook 07 computes
$\tau$ along skewers through the same snapshot.

**Requires:** a simulation snapshot, and the atomic table (01) for the ion masses.  
**Independent of:** the skewers, and the toy fields (04), which build the same cells.

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import logging; logging.getLogger("jax._src.xla_bridge").setLevel(logging.CRITICAL)
import numpy as np, jax, jax.numpy as jnp
jax.config.update("jax_enable_x64", True)          # SkewerJAX follows this flag; it never sets it itself
import matplotlib.pyplot as plt
from skewerjax.utils.plotting import use_style, use_backend
# use_backend()      # interactive figures (ipympl) in a live notebook; inline when executed headless or without ipympl
use_style()          # paper style (white); use_style("dark") for the dark one: same colour-blind-safe palette in both
from skewerjax.utils.plotting import (ENTITY_COLORS as EC, MODE_STYLES as MS, PALETTE as TOL, FG, BG, DIM, CMAPS, LABELS,
                                      FIGSIZES, add_panel_letters, plot_dag, plot_spectra, zoom_inset)

In [ ]:
plot_dag(('lines', 'simulation', 'fields',)); plt.show()

In [ ]:
import matplotlib.colors as mpl_colors
from skewerjax.simulations import ArepoHDF5Reader, HI, Ion, gas_physics as gp
DATA = os.environ.get("SKEWERJAX_DATA", "/home/STORAGE")      # folder with TNG50-4/ (docs/installation.md)
reader = ArepoHDF5Reader(f"{DATA}/TNG50-4", 25)
H = reader.header
print(f"z = {H.redshift:.4f}, a = {H.a:.6f}, box = {H.box_cMpc:.4f} cMpc, {H.n_gas:,} gas cells in {H.n_files} files, "
      f"a H(z) L = {H.vmax_kms:.1f} km/s")

## 1 · What the optical depth needs, and how the snapshot provides it

Notebook 07 derives the sum that the code evaluates from the integral along the path (review notes, Eq.
`tau_straight_los`). The result: the path is partitioned into **pieces** $p$, fragments within which the gas properties are
constant: the parts of the path inside one cell (and, by default, inside one dual segment $w_i$). Every piece contributes its own profile:

$$
\tau_j(v_\dagger) = \mathcal{C}_j \int \frac{n_{\rm com}(x)}{a(x)^{2}}\,\phi_j(v_\dagger, x)\,dx
\;\simeq\; \mathcal{C}_j \sum_p \frac{n_{{\rm com},p}\,\ell_p}{a_p^{2}}\,\phi_{j,p}(v_\dagger),
\qquad \sum_{p \in i} \ell_p = w_i,\qquad \mathcal{C}_j = \frac{e^2 f_j}{4\varepsilon_0 m_e c}.
$$

Per piece $p$ of gas on the path, the integrator therefore needs:

- the comoving density $n_{\rm com}$;
- the temperature $T$, which sets the width $b = \sqrt{2k_BT/m}$;
- the velocity along the path, $v_\parallel$, which shifts the line centre;
- the scale factor $a$ at the piece centroid, the Hubble flow from the skewer's $a(x)$;
- the length $\ell_p$.

The snapshot provides the gas in five steps, the last of which gives the container of notebook 04:

| step | code | per cell |
|---|---|---|
| 1. read | `readers.ArepoHDF5Reader.to_gas_cells` | positions [cMpc] = `Coordinates` $\times$ ckpc$/h$; $\rho$ [g cm$^{-3}$, physical] = `Density` $\times$ unit; $u$ [erg/g] = `InternalEnergy` $\times$ (km/s)$^2$; $v_{\rm pec}$ [km/s] = `Velocities` $\times\sqrt{a}$; $X_k$ = `GFM_Metals`; $n_e/n_{\rm H}$; SFR; stored $x_{\rm HI}$ |
| 2. raw cells | `gas_cells.GasCells` | $V = M/\rho$ (comoving), size $V^{1/3}$ (`size_cMpc`) |
| 3. physics | `gas_physics.ion_properties` | $n_{\rm H}$, $T$, $x_{\rm HI}$, metal ion fractions $f_{\rm ion}$ (§2); $n_{\rm ion} = \rho X_{\rm elem} f_{\rm ion}/(A\, m_u)$ |
| 4. selection | `snapshot_cells.select_cells_in_region`, `selection.select_cells_near_paths` | the cells of a region, or those that can touch the paths |
| 5. cells | `snapshot_cells`, `SimulationSkewers.to_cells` | `core.Cells`: the generating points and volumes, and per ion $n_{\rm com} = n_{\rm ion}\, a_{\rm snap}^3$, $T$ and $\mathbf{v}_{\rm pec}$ (§3) |

The rest is the core's, the same for the cells of a toy model (notebook 07): the deposition cuts the path into the
pieces of the cells, with exact lengths $\ell_p$ and centroids $s_p$; every piece takes its cell's $n_{\rm com}$ and $T$,
its velocity along the path $v_{\parallel,p} = \mathbf{v}_{\rm pec}\cdot\hat t$, and $a(s_p)$ from the skewer's $a(x)$.

## 2 · Per-cell physics (layer 3): the ingredients of the ion densities

Every ion density is

$$
n_{\rm ion} = \frac{\rho\, X_{\rm elem}}{A_{\rm elem}\, m_u}\, f_{\rm ion},
$$

with the cell's mass density $\rho$, the mass fraction $X_{\rm elem}$ of the element (`GFM_Metals`), its atomic weight
$A_{\rm elem}$ and the ion fraction $f_{\rm ion}$ (`gas_physics.ion_properties`). The ingredients:

- **Hydrogen density.** $n_{\rm H} = \rho X_{\rm H}/(A_{\rm H} m_u)$.
- **Temperature.** $T = (\gamma - 1)\,\mu\, m_p\, u/k_B$, with:
  - the internal energy $u$ and $\gamma = 5/3$;
  - the mean molecular weight $\mu = 4/(1 + 3X + 4X x_e)$, where $X = 0.76$ and $x_e = n_e/n_{\rm H}$ is the stored
    electron abundance (`gas_physics.temperature`).
- **H I fraction.** TNG stores $x_{\rm HI}$ from its own ionisation equilibrium, with the FG09 UV background, the local
  AGN radiation and the Rahmati et al. (2013) self-shielding factor.
  - SkewerJAX keeps the stored value, except in star-forming cells.
  - There it follows from the effective temperature of the two-phase medium, so those cells are recomputed at
    $T = 10^4$ K, the cold phase (`neutral_fraction`, model `"stored_sf"`).
  - The recomputation is Rahmati's photoionisation equilibrium:

$$
\frac{\Gamma_{\rm phot}}{\Gamma_{\rm UVB}} = 0.98\left[1+\left(\frac{n_{\rm H}}{n_{\rm SSh}}\right)^{1.64}\right]^{-2.28} + 0.02\left[1+\frac{n_{\rm H}}{n_{\rm SSh}}\right]^{-0.84},
\qquad
n_{\rm SSh} = 6.73\times10^{-3}\left(\frac{\sigma_\nu}{2.49\times10^{-18}}\right)^{-2/3}\left(\frac{T}{10^4}\right)^{0.17}\left(\frac{\Gamma_{\rm UVB}}{10^{-12}}\right)^{2/3}\left(\frac{f_{\rm bar}}{0.17}\right)^{-1/3}\ {\rm cm^{-3}} .
$$

  The neutral fraction $x$ then solves $\alpha_A (1 - x)^2 n_{\rm H} = x\,[\Gamma_{\rm phot} + \Gamma_{\rm coll}(1 - x)\, n_{\rm H}]$,
  with case-A recombination $\alpha_A(T)$, collisional ionisation $\Gamma_{\rm coll}(T)$ and $n_e = n_{\rm HII}$
  (`rahmati_neutral_fraction`):

$$
x = \frac{B - \sqrt{B^2 - 4 A \alpha_A}}{2A},\qquad A = \alpha_A + \Gamma_{\rm coll},\qquad B = 2\alpha_A + \frac{\Gamma_{\rm phot}}{n_{\rm H}} + \Gamma_{\rm coll} .
$$

  $\Gamma_{\rm UVB}$ is the simulation's own H I photoionisation rate (`TREECOOL_fg_dec11` at the snapshot redshift).
- **Metal ion fractions.** $f_{\rm ion} = \mathrm{CLOUDY}(n_{\rm H}, T)$ is a precomputed table of ionisation
  equilibrium: FG11 UV background, attenuated above 13.6 eV by the same self-shielding factor, and 1/10 solar
  abundances.
  - The table is interpolated linearly in redshift and bilinearly in $(\log n_{\rm H}, \log T)$ of $\log f$
    (`CloudyIonTable.fraction`).
  - CLOUDY's density coordinate is the hydrogen density, so the cell's $n_{\rm H}$ enters.

**One temperature for every species.** The same $T$ enters every ion, both in the widths, $b = \sqrt{2k_BT/(A\, m_u)}$,
and in the ion fractions.

- The hydrodynamics is single-fluid: ions, neutrals and electrons share one kinetic temperature, because collisions
  equilibrate them much faster than the gas evolves.
- What differs per species is the mass, $b_{\rm Si}/b_{\rm H} = \sqrt{1.008/28.09} = 0.19$, and the ionisation model.
- The exception is star-forming gas.
  - H I is recomputed for its cold phase ($10^4$ K), while its width and the Si III fraction still use the effective
    temperature ($1.5\times10^4$ to $2.4\times10^5$ K here).
  - A per-phase temperature would narrow these lines and change their Si III fraction; it is not done here.

In [ ]:
cells = reader.to_gas_cells(reader.read_rows(0, ["Coordinates", "Masses", "Density", "InternalEnergy", "ElectronAbundance",
                                                    "NeutralHydrogenAbundance", "GFM_Metals", "StarFormationRate", "Velocities"], stop=400_000))
SI3 = Ion("Si", 3)
P_HI, P_Si = gp.ion_properties(cells, HI), gp.ion_properties(cells, SI3)
sf = np.asarray(cells.fields["sfr"]) > 0
table = gp.CloudyIonTable(H.redshift)
from skewerjax.utils.plotting import plot_nH_T_fractions
n_thr = float(np.asarray(P_HI["n_H"])[sf].min())
fig, axs, info = plot_nH_T_fractions(np.asarray(P_HI["n_H"]), np.asarray(P_HI["T"]), np.asarray(P_HI["ion_fraction"]),
                                     np.asarray(P_Si["ion_fraction"]), sf, (table.log_dens, table.log_temp, table.log_fraction_grid(SI3)),
                                     n_thr=n_thr)
add_panel_letters(axs); fig.suptitle("400,000 TNG50-4 cells in the n_H–T plane: H I (stored equilibrium) and Si III (CLOUDY)", fontsize=22)
plt.show()
eos = info["eos"]
print(f"{sf.sum():,} star-forming cells: n_H >= {n_thr:.3f} cm^-3; "
      f"log T = {eos['slope']:.3f} log n_H + {eos['intercept']:.2f}, scatter {eos['scatter']:.3f} dex")
print(f"median x_HI of star-forming cells: stored {np.median(np.asarray(cells.fields['x_HI_stored'])[sf]):.3f}, "
      f"cold phase (used) {np.median(np.asarray(P_HI['ion_fraction'])[sf]):.3f}; CLOUDY Si III maximum: log f = {info['cloudy_max']['log_f']:.2f} "
      f"at log T = {info['cloudy_max']['log_T']:.2f}")

*Reading the figure.* The contours count cells (1, 10, 100 and 1000 per bin), the same in both panels.

- **The sharp cut near $n_{\rm H} \approx 10^{-1}\ \mathrm{cm^{-3}}$** is TNG's star-formation threshold (printed
  above). Above it, the gas is put on the Springel & Hernquist effective equation of state: its stored temperature is
  the pressure of a two-phase medium, a single function $T(n_{\rm H})$.
- **The thin strip to the right** is that equation of state. The star-forming cells lie on
  $\log T \approx 0.43 \log n_{\rm H} + 4.6$ with a scatter of 0.03 dex (dashed line; the fit is printed above).
- **Below the threshold**, gas cools down to the floor of photo-heated gas near $10^4$ K, the lower edge of the
  distribution. Shock-heated halo gas fills the upper part ($10^5$ to $10^7$ K), where $x_{\rm HI} \ll 1$ and almost no
  Si III survives.
- **(b)** Si III is a narrow ridge of the CLOUDY table near $10^{4.5}$ K at these densities. Collisional ionisation
  removes it above, and photoionisation at low $n_{\rm H}$. So only a small part of the cells carries Si III, mostly
  dense gas between $10^4$ and $10^5$ K.

## 3 · The snapshot as gas cells

The gas physics of §2 gives every cell of the snapshot what the optical depth needs. `snapshot_cells(reader, species,
lo, hi, margin=...)` turns the cells of a region of the box into `core.Cells`, the container of notebook 04, with no
skewer involved: it streams the snapshot once, keeps the cells whose generating points lie in the region, applies the
gas physics and stores, per absorbing species:

- `number_density`: the ion density $n_{\rm ion}$ of §2, made comoving, $n_{\rm com} = n_{\rm ion}\, a_{\rm snap}^3$
  [cm$^{-3}$];
- `temperature` [K] and `velocity` (the peculiar velocity, km/s): one temperature for every species (§2);
- the generating points, the cell volumes $V = M/\rho$ (so the size $V^{1/3}$ is the one the selection and the top-hat
  option use) and the periodic box.

Here the region is a cube of 10 cMpc around the densest gas of the snapshot, a halo with its filaments, for H I and
Si III. A margin of 1 cMpc around it keeps every cell whose generating point can be the nearest one to a point of the
region (the largest cells here are 0.4 cMpc), so the tessellation inside the region is exact.

For the optical depth only the cells within reach of the skewers matter: `SimulationSkewers` selects them while it
streams the snapshot (`select_cells_near_paths`) and `to_cells` gives them as the same container, with the same values
for the same cells (notebook 07).

In [ ]:
import time
from skewerjax.simulations import snapshot_cells
L = H.box_cMpc
centre, side, margin = np.array([10.5, 36.0, 34.0]), 10.0, 1.0      # the region: a cube of side 10 cMpc, and a margin around it [cMpc]
lo, hi = centre - side / 2, centre + side / 2
t0 = time.perf_counter()
gas = snapshot_cells(reader, ["HI", "SiIII"], lo, hi, margin=margin)   # core.Cells: the container of notebook 04
print(f"{len(gas):,} cells in the region and its margin, out of {H.n_gas:,} streamed, in {time.perf_counter() - t0:.1f} s")
print(gas)
for sp in gas.species_keys:
    f = gas.species[sp]
    print(f"  {sp:6s}: n_com from {f['number_density'].min():.2e} to {f['number_density'].max():.2e} cm^-3, T from {f['temperature'].min():.0f} "
          f"to {f['temperature'].max():.2e} K, |v_pec| up to {np.abs(f['velocity']).max():.0f} km/s")
print(f"cell sizes V^(1/3) from {gas.size_cMpc.min():.2g} to {gas.size_cMpc.max():.2g} cMpc (median {np.median(gas.size_cMpc):.2g})")

In [ ]:
from scipy.spatial import cKDTree
from skewerjax.utils.plotting import tol_cmap
z0, n_pix = 37.5, 700                                                     # the plane z = z0 through the region
u, w = np.linspace(lo[0], hi[0], n_pix), np.linspace(lo[1], hi[1], n_pix)
U, W = np.meshgrid(u, w, indexing="ij")
plane = np.stack([U.ravel(), W.ravel(), np.full(U.size, z0)], -1)
owner = cKDTree(gas.positions, boxsize=L).query(np.mod(plane, L), workers=-1)[1].reshape(U.shape)   # every pixel: its nearest generating point in 3D
edge = (np.diff(owner, axis=0, prepend=owner[:1]) != 0) | (np.diff(owner, axis=1, prepend=owner[:, :1]) != 0)   # the cell boundaries on the plane
ext = (lo[0], hi[0], lo[1], hi[1])
on_plane = lambda f: np.asarray(f)[owner].T                               # a per-cell field on the plane (imshow layout)
def log_n(sp):                                                            # log10 n on the plane; colours over 12 decades below its maximum
    x = np.log10(np.maximum(on_plane(gas.species[sp]["number_density"]), 1e-300))
    return x, dict(vmin=max(np.percentile(x, 1), x.max() - 12.0), vmax=x.max())
fig, axs = plt.subplots(2, 3, figsize=(24, 15), layout="constrained")
slab = (np.abs(gas.positions[:, 2] - z0) < 0.2) & np.all((gas.positions[:, :2] >= lo[:2]) & (gas.positions[:, :2] < hi[:2]), 1)
sc = axs[0, 0].scatter(gas.positions[slab, 0], gas.positions[slab, 1], c=np.log10(gas.size_cMpc[slab]), s=4, lw=0, cmap=CMAPS["sequential"])
fig.colorbar(sc, ax=axs[0, 0], label=r"$\log_{10}$ cell size $V^{1/3}$ [cMpc]", shrink=0.85)
zx, zy, zs = 12.0, 36.5, 3.0                                              # a 3 cMpc square at the edge of the filaments: single cells
axs[0, 0].add_patch(plt.Rectangle((zx, zy), zs, zs, fill=False, ec=FG, lw=1.6, ls="--"))
axs[0, 0].set(title="generating points near the plane")
x_HI, r_HI = log_n("HI")
im = axs[0, 1].imshow(x_HI, origin="lower", extent=ext, cmap=tol_cmap("iridescent"), interpolation="nearest", **r_HI)
axs[0, 1].imshow(np.ma.masked_where(~edge.T, edge.T), origin="lower", extent=ext, cmap=mpl_colors.ListedColormap([FG]), interpolation="nearest")
cut = np.unique(owner)                                                    # the cells cut by the plane, and their generating points
axs[0, 1].scatter(gas.positions[cut, 0], gas.positions[cut, 1], s=10, color=TOL["red"], lw=0)
axs[0, 1].set(xlim=(zx, zx + zs), ylim=(zy, zy + zs), title="the cells cut by the plane (zoom)")
fig.colorbar(im, ax=axs[0, 1], label=r"$\log_{10} n_{\rm HI}$ [comoving cm$^{-3}$]", shrink=0.85)
im = axs[0, 2].imshow(np.log10(on_plane(gas.species["HI"]["temperature"])), origin="lower", extent=ext, cmap=tol_cmap("incandescent"),
                      interpolation="nearest")
vq = gas.species["HI"]["velocity"][owner[::35, ::35]]                     # the in-plane peculiar velocity, every 35th pixel
axs[0, 2].quiver(U[::35, ::35], W[::35, ::35], vq[..., 0], vq[..., 1], color=FG, scale=6000, width=0.003)
fig.colorbar(im, ax=axs[0, 2], label=r"$\log_{10} T$ [K]", shrink=0.85)
axs[0, 2].set(title="temperature and velocity")
for ax, sp in ((axs[1, 0], "HI"), (axs[1, 1], "SiIII")):
    x, r = log_n(sp)
    im = ax.imshow(x, origin="lower", extent=ext, cmap=tol_cmap("iridescent"), interpolation="nearest", **r)
    fig.colorbar(im, ax=ax, label=r"$\log_{10} n_{\rm com}$ [cm$^{-3}$]", shrink=0.85)
    ax.set(title=f"{sp} density")
for ax in list(axs[0]) + list(axs[1, :2]):
    ax.set(xlabel="x [cMpc]", ylabel="y [cMpc]", aspect="equal")
for sp, sty in (("HI", MS["default"]), ("SiIII", MS["matched"])):
    n = gas.species[sp]["number_density"]
    axs[1, 2].hist(np.log10(n[n > 0]), bins=60, histtype="step", lw=2, color=EC[sp], label=sp, **{k: v for k, v in sty.items() if k == "ls"})
axs[1, 2].set(xlabel=r"$\log_{10} n_{\rm com}$ [cm$^{-3}$]", ylabel="cells", yscale="log", title="every cell of the container")
axs[1, 2].legend()
add_panel_letters(axs)
fig.suptitle(f"A {side:g} cMpc region of TNG50-4 as gas cells, on the plane z = {z0:g} cMpc: every pixel takes the cell of its nearest generating point",
             fontsize=21)
plt.show()
print(f"{cut.size:,} cells cut by the plane; {slab.sum():,} generating points within 0.2 cMpc of it")
si = gas.species["SiIII"]["number_density"]
print(f"Si III above 1e-3 of its maximum in {np.mean(si > 1e-3 * si.max()):.1%} of the cells")

*Reading the figure.*

- **(a)** The generating points within 0.2 cMpc of the plane crowd where the gas is dense: the cells are small in the halo and the filaments and
  large in the voids, a Voronoi mesh that follows the gas, unlike the regular grid of notebook 04. The dashed square
  is the zoom of (b).
- **(b)** The tessellation of the plane is exact: every pixel takes the cell whose generating point is nearest in 3D,
  which is the definition of a Voronoi cell, so the cell boundaries (lines) are where that point changes. The cells are
  cut by the plane, so their generating points (dots) lie off it and need not fall inside their own section.
- **(c)** One temperature for every species: shock-heated gas of $10^6$ to $10^7$ K around the halo, a few $10^4$ K in
  the filaments, and below $10^4$ K in the voids, cooled by their expansion. The arrows are the peculiar velocities in
  the plane, pointing mostly towards the halo.
- **(d), (e), (f)** Every species has its own densities on the same cells (colours over 12 decades below the
  maximum). Si III is present only in a small part of them (printed above), small dense knots of gas between $10^4$
  and $10^5$ K (§2) at the edge of the halo and in the filaments; in the hot halo and in the diffuse gas it is many
  decades lower, down to the floor of the CLOUDY table (the left peaks of (f)).

**This is the object that notebook 04 builds from a Gaussian random field** (`grf_cells`). Notebook 07 takes it from
here: the selection of the cells along the skewers, their deposition into the pieces of the paths, and the sum of the
pieces' profiles into $\tau$; the same functions run for both sources.

**Options.**

- `ExtractionOptions` (applied in notebook 07):
  - `deposition`: `"voronoi"` (default: the cell pieces of every dual segment), `"tophat"` or `"voronoi_crossings"`
    (the exact cell crossings, each split into equal sub-pieces);
  - `mass_unit`: `"amu"` or `"proton"`;
  - `T_constants` and `b_constants`: the physical constants of $T$ and $b$;
  - `x_HI_model`: `"stored_sf"` (default), `"stored"`, `"rahmati_all"`, ...;
  - `cloudy_density` and `cloudy_interpolation`: the density coordinate of the CLOUDY table, and `"linear"` or
    `"cubic"` interpolation.
- `IntegrationOptions`:
  - `frame`: `"exact"` (default) or `"snapshot"`;
  - `normalisation`;
  - `detuning`: `"exact"` (default) or `"linear"`.

Periodicity is not one of them: a periodic spectrum is the opt-in `velocity_period` of `core.compute_optical_depth`,
which `SimulationSkewers.tau` and `optical_depth` pass through (default: none; notebook 07, §2).

The non-default values reproduce the conventions of other codes; notebook 09 compares them. Checks:
`scripts/validation/check_tng_fields.py`, `check_tng_optical_depth.py`.